# HAPLO-CARE — chr22 real-data pilot, 5 Mb (GRCh38)

**Five 1-Mb windows**, sparse common SNPs (up to ~200 per window), 180 selected individuals; model comparison **ML + graph vs population reference + graph**. This evaluates agreement with the 1000 Genomes published statistical phasing, **not** independent true-phase accuracy.

Download the two files linked in `data/README.md` and place them in this repository's `data/` directory. **Do not unzip the `.vcf.gz`.**

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
DATA = ROOT / 'data'
DATA.mkdir(exist_ok=True)
VCF = DATA / '1kGP_high_coverage_Illumina.chr22.filtered.SNV_INDEL_SV_phased_panel.vcf.gz'
META = DATA / '20130606_g1k_3202_samples_ped_population.txt'
print('Working folder:',ROOT)
print('Python:',sys.executable)
for f in (VCF,META):
    print(f.name, 'FOUND' if f.exists() else 'MISSING', f'{f.stat().st_size/1024/1024:.1f} MB' if f.exists() else '')
assert (ROOT/'run_real_chr22_5mb.py').is_file(), 'Open notebook from the repository root.'
assert VCF.exists() and META.exists(), 'Copy both downloaded data files into the data folder, then rerun this cell.'
assert VCF.stat().st_size > 100_000_000, 'VCF file is incomplete or has the wrong name.'


## Environment

If the next cell imports NumPy, pandas, scipy and scikit-learn successfully, **do not install anything**. In particular, do not upgrade NumPy or Matplotlib in the Anaconda base environment.

In [ ]:
import numpy, pandas, scipy, sklearn
print('numpy',numpy.__version__,'pandas',pandas.__version__,'scikit-learn',sklearn.__version__,'scipy',scipy.__version__)


## Run the pilot

The code scans the compressed VCF without `pysam` or `bcftools`, extracts the five windows, freezes an individual-held-out split, fits a model independently per window, and writes donor-level results. **No matplotlib is needed.** Estimate: a few minutes to over an hour depending on storage, disk speed and Windows environment; this is not a measured runtime.

In [ ]:
import subprocess
cmd = [sys.executable, '-u', str(ROOT/'run_real_chr22_5mb.py'),
       '--vcf',str(VCF),'--panel',str(META),
       '--output',str(ROOT/'results'/'real_chr22_5mb_run'),
       '--start','30000000','--n-per-group','60','--bin-bp','5000',
       '--max-step','3','--seed','20261018']
print('Starting chr22:30,000,000–34,999,999')
subprocess.run(cmd,cwd=ROOT,check=True)


## Review and zip the generated outputs

In [ ]:
import json
import pandas as pd
from IPython.display import display
O=ROOT/'results'/'real_chr22_5mb_run'
summary=json.loads((O/'summary.json').read_text())
print('Held-out donors evaluated:',summary['n_evaluated'])
print('Reference error:',round(summary['mean_reference_error']*100,3),'%')
print('ML + graph error:',round(summary['mean_ml_error']*100,3),'%')
print('ML relative improvement:',round(summary['mean_improvement']/summary['mean_reference_error']*100,3),'%')
print('Bootstrap CI of absolute improvement (pp):',
      [round(x*100,3) for x in [summary['bootstrap_ci95_lower'],summary['bootstrap_ci95_upper']]])
display(pd.DataFrame(summary['per_window']))
display(pd.DataFrame(summary['per_population']))


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
OUTZIP=ROOT/'HAPLO_CARE_REAL_CHR22_5MB_RESULTS.zip'
assert (O/'DONE.txt').exists(), 'Pipeline has not completed all five windows.'
with ZipFile(OUTZIP,'w',ZIP_DEFLATED) as z:
    for path in sorted(O.glob('*')):
        if path.is_file():z.write(path,arcname=path.name)
print('READY:',OUTZIP,'size',round(OUTZIP.stat().st_size/1024,1),'KB')
from IPython.display import FileLink
display(FileLink(str(OUTZIP)))
